# NLP Pipeline Training
## Train and evaluate the NLP pipeline on cybersecurity attack data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nlp_pipeline_module import NLPPipeline
import json
from datetime import datetime
import os

## PipelineTrainer Class

In [ ]:
class PipelineTrainer:
    """Manages training, evaluation, and reporting of the NLP pipeline"""
    
    def __init__(self, data_path='./cybersecurity_attacks.csv', output_dir='./results'):
        """
        Initialize the trainer
        
        Args:
            data_path: Path to CSV file
            output_dir: Directory for saving results
        """
        self.data_path = data_path
        self.output_dir = output_dir
        self.df = None
        self.pipeline = None
        self.results = None
        
        # Create output directory
        os.makedirs(output_dir, exist_ok=True)
    
    def load_data(self):
        """Load and inspect data"""
        print("Loading data...")
        try:
            self.df = pd.read_csv(self.data_path, encoding='latin-1')
            print(f"✓ Data loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
            print(f"\nColumns: {self.df.columns.tolist()}")
            print(f"\nFirst few rows:")
            print(self.df.head())
            print(f"\nData types:\n{self.df.dtypes}")
            print(f"\nMissing values:\n{self.df.isnull().sum()}")
            return True
        except FileNotFoundError:
            print(f"✗ Error: File not found at {self.data_path}")
            return False
        except Exception as e:
            print(f"✗ Error loading data: {e}")
            return False

## Data Analysis

In [ ]:
    def analyze_data(self):
        """Analyze data distribution and characteristics"""
        print("\n" + "="*50)
        print("DATA ANALYSIS")
        print("="*50)
        
        # Find text and label columns (common naming patterns)
        text_candidates = [col for col in self.df.columns if 'text' in col.lower() or 'desc' in col.lower() or 'attack' in col.lower() and 'type' not in col.lower()]
        label_candidates = [col for col in self.df.columns if 'label' in col.lower() or 'type' in col.lower() or 'class' in col.lower()]
        
        print(f"\nDetected text column candidates: {text_candidates}")
        print(f"Detected label column candidates: {label_candidates}")
        
        # Try common naming patterns
        text_col = None
        label_col = None
        
        for col in self.df.columns:
            if col.lower() in ['description', 'text', 'content', 'message']:
                text_col = col
            if col.lower() in ['attack_type', 'type', 'label', 'class', 'category']:
                label_col = col
        
        if text_col and label_col:
            print(f"\nUsing columns: text='{text_col}', label='{label_col}'")
            print(f"\nLabel distribution:")
            print(self.df[label_col].value_counts())
            return text_col, label_col
        
        return None, None

## Model Training

In [ ]:
    def train(self, text_col, label_col, max_features=500, test_size=0.2):
        """Train the NLP pipeline"""
        print("\n" + "="*50)
        print("TRAINING")
        print("="*50)
        
        # Initialize pipeline
        self.pipeline = NLPPipeline(
            max_features=max_features,
            test_size=test_size,
            random_state=42
        )
        
        print(f"\nTraining on {len(self.df)} samples...")
        print(f"- Text column: {text_col}")
        print(f"- Label column: {label_col}")
        print(f"- Max features: {max_features}")
        print(f"- Test size: {test_size}")
        
        try:
            self.results = self.pipeline.fit(self.df[text_col], self.df[label_col])
            
            print("\n✓ Training completed!")
            print(f"\nModel Performance:")
            print(f"- Train size: {self.results['train_size']}")
            print(f"- Test size: {self.results['test_size']}")
            print(f"- Accuracy: {self.results['accuracy']:.4f}")
            print(f"\nClassification Report:\n{self.results['report']}")
            
            return True
        except Exception as e:
            print(f"✗ Error during training: {e}")
            import traceback
            traceback.print_exc()
            return False

## Evaluation & Feature Importance

In [ ]:
    def evaluate(self):
        """Evaluate model performance"""
        print("\n" + "="*50)
        print("EVALUATION")
        print("="*50)
        
        if self.results is None:
            print("No training results available. Run train() first.")
            return
        
        print(f"\nAccuracy: {self.results['accuracy']:.4f}")
        print(f"\nConfusion Matrix:")
        print(self.results['confusion_matrix'])
    
    def get_feature_importance(self, top_n=15):
        """Get and visualize feature importance"""
        print("\n" + "="*50)
        print("FEATURE IMPORTANCE")
        print("="*50)
        
        if self.pipeline is None:
            print("Pipeline not trained yet.")
            return
        
        try:
            features = self.pipeline.get_feature_importance(top_n=top_n)
            print(f"\nTop {top_n} important features:")
            for i, (feature, importance) in enumerate(features, 1):
                print(f"{i:2d}. {feature:20s} - {importance:.4f}")
            return features
        except Exception as e:
            print(f"Error getting feature importance: {e}")
            return None

## Make Predictions

In [ ]:
    def make_predictions(self, test_samples):
        """Make predictions on test samples"""
        print("\n" + "="*50)
        print("SAMPLE PREDICTIONS")
        print("="*50)
        
        if self.pipeline is None:
            print("Pipeline not trained yet.")
            return
        
        try:
            predictions = self.pipeline.predict(test_samples)
            probabilities = self.pipeline.predict_proba(test_samples)
            
            print(f"\nMaking predictions on {len(test_samples)} samples:\n")
            
            for i, (sample, pred, probs) in enumerate(zip(test_samples, predictions, probabilities), 1):
                print(f"Sample {i}: {sample[:80]}...")
                print(f"Prediction: {pred}")
                print(f"Probabilities:")
                for class_label, prob in zip(self.pipeline.get_classes(), probs):
                    print(f"  - {class_label}: {prob:.4f}")
                print()
            
            return predictions, probabilities
        except Exception as e:
            print(f"Error making predictions: {e}")
            import traceback
            traceback.print_exc()
            return None, None

## Full Pipeline Execution

In [ ]:
# Initialize trainer
trainer = PipelineTrainer(
    data_path='./cybersecurity_attacks.csv',
    output_dir='./results'
)

# Load data
if trainer.load_data():
    # Analyze data
    text_col, label_col = trainer.analyze_data()
    
    if text_col and label_col:
        # Train model
        if trainer.train(text_col, label_col):
            # Evaluate
            trainer.evaluate()
            
            # Feature importance
            trainer.get_feature_importance(top_n=15)

## Sample Predictions

In [ ]:
# Make predictions on sample texts
sample_texts = [
    "sql injection attack detected",
    "normal network activity",
    "suspicious login attempt",
    "malware detected in system"
]

trainer.make_predictions(sample_texts)